# Export per l'app WSA (import CSV)

**Obiettivo:** scrivere i dati puliti nei tre formati dell'importatore dell'app (`buby-rent/api/app/import_csv.py`, esempi in `buby-rent/data/`), così il team WSA li carica senza conversioni con `python -m app.import_csv <veicoli|prenotazioni|segnalazioni> <file>`.

| File | Colonne | Origine |
| --- | --- | --- |
| `veicoli.csv` | `sede, citta, tipo, targa, stato, km` | `files/flotta.csv`, `files/sedi.csv` |
| `prenotazioni.csv` | `email_cliente, nome_cliente, targa, inizio, fine, stato` | `dati_puliti/noleggi_tutti_pulito.csv` |
| `segnalazioni.csv` | `email_mittente, oggetto, testo, ricevuta_il, targa, categoria, urgenza, sintesi, targa_estratta` | `dati_puliti/comunicazioni_tutti_pulito.csv` |

Regole:
- L'app riconosce i veicoli dalla `targa`. Le e-bike non hanno targa: usano il `codice_asset` (`EB-TOR-001`).
- `km` è il chilometraggio attuale, cioè il massimo tra `km_iniziali` e i `km_fine` dei noleggi.
- Email del cliente mancante o non valida: si recupera dagli altri noleggi dello stesso `cliente_id`.
- Ora mancante: l'inizio parte alle 00:00 e la fine è alle 23:59 dello stesso giorno.
- Le righe che l'importatore scarterebbe (email non valida, veicolo sconosciuto, fine non successiva all'inizio, testo o data mancanti) non vengono esportate; il notebook le conta.
- Le segnalazioni sono i messaggi dei clienti collegati a un noleggio. Le note di tecnici e operatori restano fuori finché il contratto dati non dice dove vanno.
- L'importatore unisce le righe di un testo su più righe senza spazio: gli a capo dei testi diventano spazi.
- `categoria`, `urgenza`, `sintesi` e `targa_estratta` restano vuote finché non c'è la componente di classificazione. **Non importare `segnalazioni.csv` prima**: l'app assegnerebbe a tutte la categoria predefinita `guasto`.

In [1]:
import os
import numpy as np
import pandas as pd

BASE_FILES = "../../files"
BASE_PULITI = "../../dati_puliti"
OUT = "../../export_app"
EMAIL = r'^[^@\s]+@[^@\s]+\.[^@\s]+$'  # stessa regola di import_csv.py

sedi = pd.read_csv(f"{BASE_FILES}/sedi.csv")
flotta = pd.read_csv(f"{BASE_FILES}/flotta.csv")
nol = pd.read_csv(f"{BASE_PULITI}/noleggi_tutti_pulito.csv", dtype={'telefono_cliente': 'string', 'piva_cliente': 'string'})
com = pd.read_csv(f"{BASE_PULITI}/comunicazioni_tutti_pulito.csv")
os.makedirs(OUT, exist_ok=True)


def esporta(df, nome, scarti):
    df.to_csv(f"{OUT}/{nome}.csv", index=False)
    motivi = ', '.join(f"{motivo}: {int(n)}" for motivo, n in scarti.items() if n) or 'nessuna'
    print(f"{nome}.csv: {len(df)} righe esportate; scartate -> {motivi}")


# --- veicoli ---
targa_di = pd.Series(flotta['targa'].fillna(flotta['codice_asset']).values, index=flotta['id_veicolo'])
nome_sede = sedi.set_index('id_sede')['nome_sede'].str.replace(r'^Buby Rent ', '', regex=True)
km_fine_max = flotta['id_veicolo'].map(nol.groupby('id_veicolo')['km_fine'].max()).fillna(0)
veicoli = pd.DataFrame({
    'sede': flotta['id_sede'].map(nome_sede),
    'citta': flotta['id_sede'].map(sedi.set_index('id_sede')['comune']),
    'tipo': flotta['tipo_mezzo'].replace({'e-bike': 'ebike'}),
    'targa': flotta['id_veicolo'].map(targa_di),
    'stato': flotta['stato'],
    'km': np.maximum(flotta['km_iniziali'], km_fine_max).astype(int),
})
esporta(veicoli, 'veicoli', {})

# --- prenotazioni ---
valida = nol['email_cliente'].fillna('').str.match(EMAIL)
email_nota = nol[valida].groupby('cliente_id')['email_cliente'].agg(lambda s: s.mode().iat[0])
email = nol['email_cliente'].where(valida, nol['cliente_id'].map(email_nota))
email_di = pd.Series(email.values, index=nol['id_noleggio'])

effettiva = nol['data_riconsegna_effettiva'].notna()
prenotazioni = pd.DataFrame({
    'email_cliente': email,
    'nome_cliente': nol['nominativo_cliente'],
    'targa': nol['id_veicolo'].map(targa_di),
    'inizio': nol['data_ritiro'] + ' ' + nol['ora_ritiro'].fillna('00:00'),
    'fine': (nol['data_riconsegna_effettiva'].fillna(nol['data_riconsegna_prevista']) + ' '
             + nol['ora_riconsegna_effettiva'].where(effettiva).fillna('23:59')),
    'stato': nol['stato_noleggio'],
})
scarti_p = {
    'email non valida': ~prenotazioni['email_cliente'].fillna('').str.match(EMAIL),
    'veicolo sconosciuto': prenotazioni['targa'].isna(),
    'fine non dopo inizio': pd.to_datetime(prenotazioni['fine']) <= pd.to_datetime(prenotazioni['inizio']),
}
da_tenere = ~np.logical_or.reduce(list(scarti_p.values()))
esporta(prenotazioni[da_tenere], 'prenotazioni', {k: v.sum() for k, v in scarti_p.items()})

# --- segnalazioni: messaggi dei clienti collegati a un noleggio ---
cli = com[(com['mittente_tipo'] == 'cliente') & com['id_noleggio_corretto'].notna()]
segnalazioni = pd.DataFrame({
    'email_mittente': cli['mittente'].where(cli['mittente'].str.contains('@', na=False),
                                            cli['id_noleggio_corretto'].map(email_di)),
    'oggetto': cli['oggetto_clean'].str.replace(r'\s*\n\s*', ' ', regex=True),
    'testo': cli['testo_clean'].str.replace(r'\s*\n\s*', ' ', regex=True),
    'ricevuta_il': cli['data_ora'],
    'targa': cli['id_noleggio_corretto'].map(nol.set_index('id_noleggio')['id_veicolo']).map(targa_di),
    'categoria': pd.NA, 'urgenza': pd.NA, 'sintesi': pd.NA, 'targa_estratta': pd.NA,
})
scarti_s = {campo + ' mancante': segnalazioni[campo].isna() for campo in ['email_mittente', 'testo', 'ricevuta_il', 'targa']}
da_tenere = ~np.logical_or.reduce(list(scarti_s.values()))
esporta(segnalazioni[da_tenere], 'segnalazioni', {k: v.sum() for k, v in scarti_s.items()})

# Vincoli dell'app (models.py e import_csv.py)
assert set(veicoli['tipo']) <= {'auto', 'furgone', 'ebike'}
assert set(veicoli['stato']) <= {'disponibile', 'prenotato', 'noleggiato', 'in_manutenzione', 'danneggiato', 'in_trasferimento'}
assert veicoli['targa'].notna().all() and veicoli['targa'].is_unique
assert set(prenotazioni['stato']) <= {'richiesta', 'confermata', 'in_corso', 'conclusa', 'annullata'}
veicoli.head()

veicoli.csv: 64 righe esportate; scartate -> nessuna
prenotazioni.csv: 4905 righe esportate; scartate -> email non valida: 88, veicolo sconosciuto: 4
segnalazioni.csv: 1388 righe esportate; scartate -> email_mittente mancante: 12, testo mancante: 1, ricevuta_il mancante: 22


,sede,citta,tipo,targa,stato,km
0,Torino Lingotto,Torino,auto,GL305PT,disponibile,104191
1,Torino Lingotto,Torino,auto,GL311RS,disponibile,70229
2,Torino Lingotto,Torino,auto,GM842BD,disponibile,66750
3,Torino Lingotto,Torino,auto,GP127FK,disponibile,46172
4,Torino Lingotto,Torino,auto,GP129LH,disponibile,60614
